In [20]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path("./")

INPUT_FILE = BASE_DIR / "tvpvar_input_scaled.csv"
OUT_FILE = BASE_DIR / "tvpvar_input_scaled_dcmetals.csv"

KEEP_COLS = [
    "Date",
    "dlog_SOLVPN",
    "dlog_COPPER",
    "dlog_GOLD",
    "dlog_SILVER",
]

df = pd.read_csv(INPUT_FILE)

missing = [c for c in KEEP_COLS if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

out = df[KEEP_COLS].copy()
out.to_csv(OUT_FILE, index=False)

print(f"Saved: {OUT_FILE}")
print(out.head())

Saved: tvpvar_input_scaled_dcmetals.csv
         Date  dlog_SOLVPN  dlog_COPPER  dlog_GOLD  dlog_SILVER
0  2020-10-13    -0.408673    -0.388006  -1.687144    -2.111589
1  2020-10-14    -1.083456     0.095267   0.536998     0.453141
2  2020-10-15    -0.060821     0.641368   0.009273    -0.353759
3  2020-10-19    -0.419849     0.329228   0.174788     0.830391
4  2020-10-20     0.082134     1.132160   0.108119     0.470602


In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# ----------------------------
# 1) Load
# ----------------------------
df = pd.read_csv("./tvpvar_input_scaled.csv")
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

gfevd = np.load("./gfevd_all.npy")

# 7개 변수 전체 사용
VAR_NAMES = [
    "dlog_SOLVPN",
    "dlog_COPPER",
    "dlog_GOLD",
    "dlog_SILVER",
    "dlog_DXY",
    "d_UST10Y",
    "dlog_VIX",
]

# ----------------------------
# 2) 기본 검증
# ----------------------------
if gfevd.ndim != 3:
    raise ValueError(f"gfevd must be 3D array, got shape={gfevd.shape}")

T_gfevd, k1, k2 = gfevd.shape
if k1 != k2:
    raise ValueError(f"gfevd must be square on last two dims, got shape={gfevd.shape}")

if k1 != len(VAR_NAMES):
    raise ValueError(
        f"VAR_NAMES length ({len(VAR_NAMES)}) != gfevd variable size ({k1}). "
        f"Current gfevd shape={gfevd.shape}. "
        f"If gfevd is 4x4, you need to recompute it with 7 variables."
    )

# ----------------------------
# 3) 날짜 길이 정렬
# ----------------------------
if len(df) > T_gfevd:
    diff = len(df) - T_gfevd
    df = df.iloc[diff:].reset_index(drop=True)
elif len(df) < T_gfevd:
    raise ValueError(
        f"Date rows ({len(df)}) < gfevd rows ({T_gfevd})."
    )

dates = df["Date"].copy()

# ----------------------------
# 4) 시각화 시작 시점
# ----------------------------
PLOT_START_DATE = "2021-01-01"
plot_mask = dates >= pd.to_datetime(PLOT_START_DATE)

# ----------------------------
# 5) TCI / NET 계산
# ----------------------------
def compute_tci(g):
    k = g.shape[1]
    diag = np.einsum("tii->t", g)
    offdiag = g.sum(axis=(1, 2)) - diag
    return offdiag / k * 100.0

def compute_net(g):
    T, k, _ = g.shape
    net = np.zeros((T, k))
    for t in range(T):
        m = g[t] * 100.0
        diag = np.diag(m)
        from_others = m.sum(axis=1) - diag
        to_others = m.sum(axis=0) - diag
        net[t] = to_others - from_others
    return net

tci = compute_tci(gfevd)
net = compute_net(gfevd)

plot_dates = dates[plot_mask].reset_index(drop=True)
plot_tci = tci[plot_mask.values]
plot_net = net[plot_mask.values]

# ----------------------------
# 6) Event 정의
# ----------------------------
events = [
    # AI / Data Center
    ("2022-11-30", "ChatGPT", "ai"),
    ("2023-01-23", "MSFT-OpenAI", "ai"),
    ("2023-03-14", "GPT-4", "ai"),
    ("2023-11-06", "OpenAI DevDay", "ai"),
    ("2024-01-01", "AI investment", "ai"),
    ("2024-03-18", "Blackwell", "ai"),
    ("2024-05-01", "Hyperscaler capex", "ai"),
    ("2024-07-31", "BigTech AI capex", "ai"),
    ("2024-10-15", "OCP Blackwell", "ai"),
    ("2025-01-21", "Stargate", "ai"),

    # Copper / Supply
    ("2021-11-15", "Copper shortage", "copper"),
    ("2023-03-01", "Copper bullish call", "copper"),
    ("2023-11-28", "Panama mine", "copper"),
    ("2024-08-15", "LME copper low", "copper"),
    ("2025-02-25", "US copper probe", "copper"),
    ("2025-03-10", "Copper policy risk", "copper"),

    # Infra / System
    ("2021-04-01", "Cloud demand", "infra"),
    ("2021-10-28", "Meta rename", "infra"),
    ("2022-03-22", "Hopper/H100", "infra"),
    ("2022-06-15", "Supply chain", "infra"),
    ("2022-09-15", "Energy spike", "infra"),
    ("2022-09-20", "H100 production", "infra"),
    ("2023-05-01", "AI infra demand", "infra"),
    ("2023-07-01", "GPU shortage", "infra"),
    ("2023-09-15", "DC deal peak", "infra"),
    ("2023-10-15", "Liquid cooling", "infra"),
    ("2024-09-20", "Power contract", "infra"),
]

COLOR_MAP = {
    "ai": "blue",
    "copper": "red",
    "infra": "green",
}

events = [(pd.to_datetime(d), label, cat) for d, label, cat in events]
events = [e for e in events if e[0] >= pd.to_datetime(PLOT_START_DATE)]
events = sorted(events, key=lambda x: x[0])

# ----------------------------
# 7) Event shape + annotation helper
# ----------------------------
def add_event_lines_and_labels(fig, events, y_min, y_max):
    yr = y_max - y_min
    if yr == 0:
        yr = 1.0
    levels = [y_max - yr * (0.04 + i * 0.09) for i in range(8)]

    for i, (dt, label, cat) in enumerate(events):
        color = COLOR_MAP[cat]

        fig.add_shape(
            type="line",
            x0=dt,
            x1=dt,
            y0=y_min,
            y1=y_max,
            line=dict(color=color, width=1.1, dash="dot")
        )

        fig.add_annotation(
            x=dt,
            y=levels[i % len(levels)],
            text=label,
            showarrow=False,
            textangle=-90,
            font=dict(size=10, color=color),
            xanchor="right",
            yanchor="top"
        )

def add_event_legend(fig):
    for cat, color in COLOR_MAP.items():
        fig.add_trace(
            go.Scatter(
                x=[None],
                y=[None],
                mode="lines",
                line=dict(color=color, width=2, dash="dot"),
                name=f"event_{cat}",
                hoverinfo="skip",
                showlegend=True
            )
        )

# ----------------------------
# 8) TCI figure
# ----------------------------
fig_tci = go.Figure()

fig_tci.add_trace(
    go.Scatter(
        x=plot_dates,
        y=plot_tci,
        mode="lines",
        name="TCI"
    )
)

tci_ymin = float(np.min(plot_tci))
tci_ymax = float(np.max(plot_tci))

add_event_lines_and_labels(fig_tci, events, tci_ymin, tci_ymax)
add_event_legend(fig_tci)

fig_tci.update_layout(
    title="TCI (SOLVPN + COPPER + GOLD + SILVER + DXY + UST10Y + VIX)",
    xaxis_title="Date",
    yaxis_title="TCI",
    height=700,
    hovermode="x unified"
)

fig_tci.show()

# ----------------------------
# 9) NET figure
# ----------------------------
fig_net = go.Figure()

for i, var in enumerate(VAR_NAMES):
    fig_net.add_trace(
        go.Scatter(
            x=plot_dates,
            y=plot_net[:, i],
            mode="lines",
            name=var
        )
    )

net_ymin = float(np.min(plot_net))
net_ymax = float(np.max(plot_net))

add_event_lines_and_labels(fig_net, events, net_ymin, net_ymax)
add_event_legend(fig_net)

fig_net.add_hline(y=0, line_width=1, line_color="black")

fig_net.update_layout(
    title="NET Spillover (7 Variables)",
    xaxis_title="Date",
    yaxis_title="NET Spillover",
    height=750,
    hovermode="x unified"
)

fig_net.show()